# 01 — Seismic data EDA

Explore the structure, missingness, distributions, and cleanup needs of HCAI’s California hospital seismic dataset. Records describe buildings and related campus structures; a facility can have many records. The source filename indicates a September 3, 2026 snapshot.

SPC describes structural performance, NPC describes nonstructural systems, and Hazus scores describe collapse probabilities conditional on design-level ground motion. See the [data dictionary](../docs/SEISMIC_DATA.md#detailed-data-dictionary) for full field definitions and [data sources](../docs/DATA_SOURCES.md#seismic-ratings-and-collapse-probabilities) for provenance.

All records remain in this analysis; raw files are unchanged.


In [1]:
import csv
import hashlib
import io
import math
import re
import statistics
from collections import Counter, defaultdict
from pathlib import Path
from IPython.display import Markdown, display

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "data/raw").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")

SOURCE = ROOT / "data/raw/seismic-ratings-and-collapse-probabilities-of-california-hospitals-.csv"


In [2]:
HAZUS = ("2007 Hazus Score (%)", "2010 Hazus Score (%)")
IN_SERVICE = "OSHPD 1-In Service"
TYPES = {
    "County Code": "Text/category; split county number and name if needed",
    "Perm ID": "Text identifier",
    "Facility Name": "Text",
    "City": "Text/category",
    "Building Nbr": "Text identifier",
    "Building Name": "Text",
    "Building Status": "Category",
    "SPC Rating": "Category; retain suffixes",
    HAZUS[0]: "Nullable numeric percentage",
    HAZUS[1]: "Nullable numeric percentage",
    "HCAI NPC Rating": "Category; retain suffixes",
    "AB 1882 Notice": "Text/category with optional notice",
    "Latitude": "Numeric coordinate",
    "Longitude": "Numeric coordinate",
    "Count": "Integer count helper",
}


In [3]:
def read_source(path):
    raw = path.read_bytes()
    reader = csv.reader(io.StringIO(raw.decode("cp1252"), newline=""))
    headers = next(reader, [])
    normalized = [header.strip() for header in headers]
    if len(set(normalized)) != len(normalized) or set(normalized) != set(TYPES):
        raise ValueError("Unexpected columns; review the source schema before profiling.")
    rows = []
    for line, values in enumerate(reader, start=2):
        if len(values) != len(headers):
            raise ValueError(f"CSV record {line} has {len(values)} fields; expected {len(headers)}.")
        rows.append(dict(zip(normalized, values)))
    if not rows:
        raise ValueError("The input CSV has no observations.")
    return raw, headers, rows


In [4]:
path = SOURCE.resolve()
raw, headers, rows = read_source(path)
input_sha256 = hashlib.sha256(raw).hexdigest()
print(f"Loaded {len(rows):,} records and {len(headers)} columns.")
print(f"Input SHA-256: {input_sha256}")


Loaded 4,690 records and 15 columns.
Input SHA-256: 054288eb1d387750790e19d3429b9622da11a019c842d9c7efdaee88904823cc


In [5]:
def number(value):
    try:
        parsed = float(value)
    except ValueError:
        return None
    return parsed if math.isfinite(parsed) else None

def blank(value):
    return not value.strip()

def inferred_type(values):
    nonblank = [v for v in values if not blank(v)]
    if not nonblank:
        return "All blank"
    if all(re.fullmatch(r"[+-]?\d+", v.strip()) for v in nonblank):
        return "Integer-like"
    if all(number(v) is not None for v in nonblank):
        return "Numeric-like"
    return "Text/mixed labels"


In [6]:
def numeric_summary(rows, column):
    supplied = [r[column] for r in rows if not blank(r[column])]
    parsed = [number(v) for v in supplied]
    finite = [v for v in parsed if v is not None]
    values = [v for v in finite if not (column == HAZUS[1] and v == -50)]
    stats = [None] * 6
    if values:
        quartiles = statistics.quantiles(values, n=4, method="inclusive") if len(values) > 1 else values * 3
        stats = [min(values), quartiles[0], statistics.median(values), quartiles[2], max(values), statistics.mean(values)]
    return {
        "values": values,
        "invalid": sum(v is None for v in parsed),
        "stats": stats,
        "zero": sum(v == 0 for v in values),
        "outside_probability": sum(v < 0 or v > 100 for v in finite),
        "fault_marker": finite.count(-50) if column == HAZUS[1] else 0,
    }


In [7]:
def profile(rows):
    counts = {column: Counter(r[column] for r in rows) for column in TYPES}
    facilities = defaultdict(list)
    names = defaultdict(set)
    coordinates = defaultdict(set)
    for row in rows:
        facilities[row["Perm ID"]].append(row)
        names[row["Facility Name"]].add(row["Perm ID"])
        coordinates[(row["Latitude"], row["Longitude"])].add(row["Perm ID"])
    city_conflicts = {
        key: Counter(r["City"] for r in group)
        for key, group in facilities.items()
        if len({r["City"] for r in group}) > 1
    }
    numeric = {c: numeric_summary(rows, c) for c in (*HAZUS, "Latitude", "Longitude", "Count")}
    coverage = Counter(tuple(not blank(r[c]) for c in HAZUS) for r in rows)
    return {
        "counts": counts,
        "facilities": facilities,
        "names": names,
        "coordinates": coordinates,
        "city_conflicts": city_conflicts,
        "numeric": numeric,
        "coverage": coverage,
    }


In [8]:
summary = profile(rows)


In [9]:
def table(headers, rows):
    def escape(value):
        return str(value).replace("|", "\\|").replace("\n", " ")
    lines = ["| " + " | ".join(map(escape, headers)) + " |", "| " + " | ".join("---" for _ in headers) + " |"]
    lines.extend("| " + " | ".join(map(escape, row)) + " |" for row in rows)
    return "\n".join(lines)

def rate(count, total):
    return f"{100 * count / total:.1f}%" if total else "—"

def fmt(value):
    return "—" if value is None else f"{value:,.4f}".rstrip("0").rstrip(".")


In [10]:
def display_sections(sections):
    display(Markdown("\n\n".join(sections)))


## Snapshot overview

The exact status `OSHPD 1-In Service` is used as a comparison subset, not the final modeling population.


In [11]:
n = len(rows)
counts = summary["counts"]
numeric = summary["numeric"]
in_service = [r for r in rows if r["Building Status"] == IN_SERVICE]
duplicate_rows = n - len({tuple(r[c] for c in TYPES) for r in rows})
duplicate_keys = n - len({(r["Perm ID"], r["Building Nbr"]) for r in rows})
present = sum(summary["coverage"][pair] for pair in [(True, False), (False, True), (True, True)])
source_path = path.relative_to(ROOT).as_posix() if path.is_relative_to(ROOT) else path.name


In [12]:
sections = [
f"- {n:,} records, {len(headers)} columns, {len(summary['facilities']):,} facility IDs, and {len(counts['County Code'])} counties.\n"
            f"- {duplicate_rows:,} exact duplicate rows and {duplicate_keys:,} duplicate facility/building key pairs.\n"
            f"- {summary['coverage'][(False, False)]:,} records ({rate(summary['coverage'][(False, False)], n)}) have neither Hazus score. Only {present:,} have at least one score.\n"
            f"- {len(in_service):,} records ({rate(len(in_service), n)}) have the exact status `{IN_SERVICE}`.\n"
            f"- {len(summary['city_conflicts'])} facility IDs have multiple city labels and require review."
]
display_sections(sections)


- 4,690 records, 15 columns, 423 facility IDs, and 56 counties.
- 0 exact duplicate rows and 0 duplicate facility/building key pairs.
- 4,075 records (86.9%) have neither Hazus score. Only 615 have at least one score.
- 3,183 records (67.9%) have the exact status `OSHPD 1-In Service`.
- 2 facility IDs have multiple city labels and require review.

## Column types and missingness

Cells are loaded as text; inferred forms and recommended types are shown below. Distinct counts exclude blanks. Keep blanks, `N/A` (not applicable), `NYA` (not yet available), and numeric zero separate. Numeric-looking IDs remain identifiers.


In [13]:
sections = [
]

schema = []

for c in TYPES:
    values = [r[c] for r in rows]
    empty = sum(blank(v) for v in values)
    schema.append([c, inferred_type(values), TYPES[c], len({v for v in values if not blank(v)}), empty, rate(empty, n), counts[c]["N/A"], counts[c]["NYA"]])

sections.append(table(["Column", "Observed form", "Recommended type", "Distinct nonblank", "Blank", "Blank %", "N/A", "NYA"], schema))



display_sections(sections)


| Column | Observed form | Recommended type | Distinct nonblank | Blank | Blank % | N/A | NYA |
| --- | --- | --- | --- | --- | --- | --- | --- |
| County Code | Text/mixed labels | Text/category; split county number and name if needed | 56 | 0 | 0.0% | 0 | 0 |
| Perm ID | Integer-like | Text identifier | 423 | 0 | 0.0% | 0 | 0 |
| Facility Name | Text/mixed labels | Text | 416 | 0 | 0.0% | 0 | 0 |
| City | Text/mixed labels | Text/category | 251 | 0 | 0.0% | 0 | 0 |
| Building Nbr | Text/mixed labels | Text identifier | 4690 | 0 | 0.0% | 0 | 0 |
| Building Name | Text/mixed labels | Text | 2856 | 0 | 0.0% | 0 | 0 |
| Building Status | Text/mixed labels | Category | 28 | 0 | 0.0% | 0 | 0 |
| SPC Rating | Text/mixed labels | Category; retain suffixes | 12 | 0 | 0.0% | 1141 | 0 |
| 2007 Hazus Score (%) | Numeric-like | Nullable numeric percentage | 159 | 4287 | 91.4% | 0 | 0 |
| 2010 Hazus Score (%) | Numeric-like | Nullable numeric percentage | 171 | 4395 | 93.7% | 0 | 0 |
| HCAI NPC Rating | Text/mixed labels | Category; retain suffixes | 13 | 0 | 0.0% | 343 | 0 |
| AB 1882 Notice | Text/mixed labels | Text/category with optional notice | 2 | 3936 | 83.9% | 0 | 0 |
| Latitude | Numeric-like | Numeric coordinate | 422 | 0 | 0.0% | 0 | 0 |
| Longitude | Numeric-like | Numeric coordinate | 422 | 0 | 0.0% | 0 | 0 |
| Count | Integer-like | Integer count helper | 1 | 0 | 0.0% | 0 | 0 |

## Numeric distributions

Summaries use finite, nonblank numbers and exclude the 2010 Hazus `-50` fault marker. Quartiles use linear interpolation; other out-of-range values are flagged below.

Hazus statistics describe only scored buildings. The two model versions are not a time series and should not be averaged. Coordinate summaries count building records, including repeated facility locations.


In [14]:
sections = [
    table(["Column", "Valid n", "Invalid/nonfinite", "Min", "Q1", "Median", "Q3", "Max", "Mean", "Zeros"], [
                [c, len(v["values"]), v["invalid"], *map(fmt, v["stats"]), v["zero"]] for c, v in numeric.items()
            ]),
    table(["Hazus column", "Outside 0–100", "Documented -50 marker"], [[c, numeric[c]["outside_probability"], numeric[c]["fault_marker"]] for c in HAZUS]),
]

display_sections(sections)


| Column | Valid n | Invalid/nonfinite | Min | Q1 | Median | Q3 | Max | Mean | Zeros |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2007 Hazus Score (%) | 403 | 0 | 0 | 0.065 | 0.31 | 0.87 | 31.75 | 1.4849 | 29 |
| 2010 Hazus Score (%) | 295 | 0 | 0 | 0.54 | 0.98 | 2.265 | 36.74 | 3.7995 | 6 |
| Latitude | 4690 | 0 | 32.6189 | 33.8997 | 34.363 | 37.7061 | 41.7745 | 35.6637 | 0 |
| Longitude | 4690 | 0 | -124.194 | -121.434 | -118.4875 | -117.8907 | -114.5951 | -119.4325 | 0 |
| Count | 4690 | 0 | 1 | 1 | 1 | 1 | 1 | 1 | 0 |

| Hazus column | Outside 0–100 | Documented -50 marker |
| --- | --- | --- |
| 2007 Hazus Score (%) | 0 | 0 |
| 2010 Hazus Score (%) | 0 | 0 |

## Hazus availability

Compare availability across model versions and building statuses. Missingness does not establish why a score is absent or justify filling it with zero.


In [15]:
sections = [
    table(["Availability", "Records", "% of all records"], [
                [label, summary["coverage"][pair], rate(summary["coverage"][pair], n)]
                for pair, label in [((False, False), "Neither"), ((True, False), "2007 only"), ((False, True), "2010 only"), ((True, True), "Both")]
            ]),
]

display_sections(sections)


| Availability | Records | % of all records |
| --- | --- | --- |
| Neither | 4075 | 86.9% |
| 2007 only | 320 | 6.8% |
| 2010 only | 212 | 4.5% |
| Both | 83 | 1.8% |

In [16]:
sections = [
    "### Coverage by building status",
    "Score percentages use each status's record count as the denominator.",
]

status_rows = []

for status, count in counts["Building Status"].most_common():
    group = [r for r in rows if r["Building Status"] == status]
    available = [sum(not blank(r[c]) for r in group) for c in HAZUS]
    status_rows.append([status, count, rate(count, n), *available, *(rate(v, count) for v in available)])

sections.append(table(["Status", "Records", "% of all", "2007 present", "2010 present", "2007 %", "2010 %"], status_rows))

display_sections(sections)


### Coverage by building status

Score percentages use each status's record count as the denominator.

| Status | Records | % of all | 2007 present | 2010 present | 2007 % | 2010 % |
| --- | --- | --- | --- | --- | --- | --- |
| OSHPD 1-In Service | 3183 | 67.9% | 380 | 245 | 11.9% | 7.7% |
| OSHPD 1-Proposed | 512 | 10.9% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 1-Under Construction | 304 | 6.5% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 1-Not an Independent Building | 169 | 3.6% | 10 | 8 | 5.9% | 4.7% |
| OSHPD 1-Equipment Yard | 156 | 3.3% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 1-Tanks | 86 | 1.8% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 1R-No Gen Acute Care - OSHPD Bldg | 79 | 1.7% | 13 | 36 | 16.5% | 45.6% |
| OSHPD 1-Tunnels | 61 | 1.3% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 1-Not a Building Structure | 45 | 1.0% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 2-Skilled Nursing Only | 29 | 0.6% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 5-Acute Psych Only | 9 | 0.2% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 5-Under Construction | 9 | 0.2% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 3 - Local-Clinic - Local Jurisdiction | 8 | 0.2% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 3 - Local-Outpatient Only | 8 | 0.2% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 3 - Local-Proposed | 7 | 0.1% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 5-Proposed | 4 | 0.1% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 1R-Skilled Nursing Only | 3 | 0.1% | 0 | 1 | 0.0% | 33.3% |
| OSHPD 1R-Acute Psych / SNF only | 3 | 0.1% | 0 | 3 | 0.0% | 100.0% |
| OSHPD 1-No Gen Acute Care - OSHPD Bldg | 3 | 0.1% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 1R-Delicensed Clinic under OSHPD | 2 | 0.0% | 0 | 1 | 0.0% | 50.0% |
| OSHPD 2-Proposed | 2 | 0.0% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 5-Not an Independent Building | 2 | 0.0% | 0 | 0 | 0.0% | 0.0% |
| Adjacent Building - Local Jurisdiction-Outpatient Only | 1 | 0.0% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 5-Acute Psych / SNF only | 1 | 0.0% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 1R-Acute Psych Only | 1 | 0.0% | 0 | 1 | 0.0% | 100.0% |
| OSHPD 1R-Tunnels | 1 | 0.0% | 0 | 0 | 0.0% | 0.0% |
| OSHPD 2-Under Construction | 1 | 0.0% | 0 | 0 | 0.0% | 0.0% |
| Local-Tanks | 1 | 0.0% | 0 | 0 | 0.0% | 0.0% |

## Rating distributions

Compare full-dataset and in-service frequencies. Retain suffixes and extended codes such as `4D`, `3R`, and `4D-L1` until their meanings are confirmed.


In [17]:
sections = [
]

for c in ("SPC Rating", "HCAI NPC Rating"):
    subcounts = Counter(r[c] for r in in_service)
    sections.extend([
        f"### {c}",
        table(["Rating", "All records", "% of all", "OSHPD 1-In Service", "% of in-service subset"], [
            [v or "(blank)", count, rate(count, n), subcounts[v], rate(subcounts[v], len(in_service))]
            for v, count in counts[c].most_common()
        ]),
    ])



display_sections(sections)


### SPC Rating

| Rating | All records | % of all | OSHPD 1-In Service | % of in-service subset |
| --- | --- | --- | --- | --- |
| 5 | 1251 | 26.7% | 1242 | 39.0% |
| N/A | 1141 | 24.3% | 1 | 0.0% |
| 4 | 718 | 15.3% | 717 | 22.5% |
| 2 | 636 | 13.6% | 634 | 19.9% |
| 5s | 451 | 9.6% | 97 | 3.0% |
| 3 | 333 | 7.1% | 333 | 10.5% |
| 4s | 61 | 1.3% | 61 | 1.9% |
| 3s | 46 | 1.0% | 46 | 1.4% |
| 4D | 29 | 0.6% | 29 | 0.9% |
| 1 | 17 | 0.4% | 17 | 0.5% |
| 2s | 6 | 0.1% | 5 | 0.2% |
| 1s | 1 | 0.0% | 1 | 0.0% |

### HCAI NPC Rating

| Rating | All records | % of all | OSHPD 1-In Service | % of in-service subset |
| --- | --- | --- | --- | --- |
| 2 | 1604 | 34.2% | 1542 | 48.4% |
| 4 | 1267 | 27.0% | 1095 | 34.4% |
| 5s | 741 | 15.8% | 15 | 0.5% |
| N/A | 343 | 7.3% | 0 | 0.0% |
| 3 | 234 | 5.0% | 230 | 7.2% |
| 5 | 208 | 4.4% | 120 | 3.8% |
| 4s | 164 | 3.5% | 59 | 1.9% |
| 4D-L1 | 75 | 1.6% | 74 | 2.3% |
| 3R | 24 | 0.5% | 24 | 0.8% |
| 2s | 23 | 0.5% | 18 | 0.6% |
| 1 | 3 | 0.1% | 3 | 0.1% |
| 4D-L2 | 3 | 0.1% | 3 | 0.1% |
| 3s | 1 | 0.0% | 0 | 0.0% |

## Seismic notices

Blank notices do not imply safety. Supplied notices may reveal a rating target, creating leakage if used as predictors.


In [18]:
sections = [
    table(["Notice", "Records", "% of all"], [[v or "(blank)", count, rate(count, n)] for v, count in counts["AB 1882 Notice"].most_common()]),
]

display_sections(sections)


| Notice | Records | % of all |
| --- | --- | --- |
| (blank) | 3936 | 83.9% |
| This building does not significantly jeopardize life, but may not be repairable or functional following an earthquake. | 636 | 13.6% |
| Earthquake Resilient | 118 | 2.5% |

## Identifier and geographic checks

ID patterns are snapshot checks. Coordinates are checked against global bounds only, not California boundaries or source addresses. Repeated names or coordinates alone do not justify deduplication.


In [19]:
sections = [
]

lat_invalid = sum(number(r["Latitude"]) is None or not -90 <= number(r["Latitude"]) <= 90 for r in rows)

lon_invalid = sum(number(r["Longitude"]) is None or not -180 <= number(r["Longitude"]) <= 180 for r in rows)

checks = [
    ["Blank facility IDs", sum(blank(r["Perm ID"]) for r in rows)],
    ["Blank building IDs", sum(blank(r["Building Nbr"]) for r in rows)],
    ["Facility IDs not matching five digits", sum(re.fullmatch(r"\d{5}", r["Perm ID"]) is None for r in rows)],
    ["Building IDs not matching BLD- plus five digits", sum(re.fullmatch(r"BLD-\d{5}", r["Building Nbr"]) is None for r in rows)],
    ["Exact duplicate rows", duplicate_rows],
    ["Duplicate facility/building key pairs", duplicate_keys],
    ["Repeated building IDs", n - len(counts["Building Nbr"])],
    ["Facility names shared by multiple facility IDs", sum(len(v) > 1 for v in summary["names"].values())],
    ["Unique coordinate pairs", len(summary["coordinates"])],
    ["Coordinate pairs shared by multiple facility IDs", sum(len(v) > 1 for v in summary["coordinates"].values())],
    ["Missing, invalid, or out-of-global-range latitude", lat_invalid],
    ["Missing, invalid, or out-of-global-range longitude", lon_invalid],
    ["Cells with surrounding whitespace", sum(v != v.strip() for r in rows for v in r.values())],
]

sections.append(table(["Check", "Count"], checks))



display_sections(sections)


| Check | Count |
| --- | --- |
| Blank facility IDs | 0 |
| Blank building IDs | 0 |
| Facility IDs not matching five digits | 0 |
| Building IDs not matching BLD- plus five digits | 0 |
| Exact duplicate rows | 0 |
| Duplicate facility/building key pairs | 0 |
| Repeated building IDs | 0 |
| Facility names shared by multiple facility IDs | 7 |
| Unique coordinate pairs | 422 |
| Coordinate pairs shared by multiple facility IDs | 1 |
| Missing, invalid, or out-of-global-range latitude | 0 |
| Missing, invalid, or out-of-global-range longitude | 0 |
| Cells with surrounding whitespace | 0 |

In [20]:
sections = [
    "### Facility-level consistency",
    table(["Field", "Facility IDs with multiple values"], [[c, sum(len({r[c] for r in group}) > 1 for group in summary["facilities"].values())] for c in ["Facility Name", "County Code", "City", "Latitude", "Longitude"]]),
    table(["Facility ID", "City labels and record counts"], [[key, "; ".join(f"{city}: {count}" for city, count in values.most_common())] for key, values in sorted(summary["city_conflicts"].items())]) if summary["city_conflicts"] else "No within-facility city conflicts found.",
]

display_sections(sections)


### Facility-level consistency

| Field | Facility IDs with multiple values |
| --- | --- |
| Facility Name | 0 |
| County Code | 0 |
| City | 2 |
| Latitude | 0 |
| Longitude | 0 |

| Facility ID | City labels and record counts |
| --- | --- |
| 11000 | Fall River Mills: 9; Burney: 1 |
| 18219 | Redwood City: 2; Redwoord City: 1 |

In [21]:
sections = [
    "### Ten counties with the most records",
    table(["County", "Building records", "% of all", "Facility IDs"], [[county, count, rate(count, n), len({r["Perm ID"] for r in rows if r["County Code"] == county})] for county, count in counts["County Code"].most_common(10)]),
    "County counts show dataset representation, not earthquake risk.",
]

display_sections(sections)


### Ten counties with the most records

| County | Building records | % of all | Facility IDs |
| --- | --- | --- | --- |
| 19 - Los Angeles | 1040 | 22.2% | 91 |
| 30 - Orange | 383 | 8.2% | 36 |
| 37 - San Diego | 367 | 7.8% | 26 |
| 36 - San Bernardino | 307 | 6.5% | 25 |
| 33 - Riverside | 219 | 4.7% | 20 |
| 43 - Santa Clara | 218 | 4.6% | 17 |
| 01 - Alameda | 161 | 3.4% | 16 |
| 34 - Sacramento | 145 | 3.1% | 14 |
| 10 - Fresno | 141 | 3.0% | 10 |
| 39 - San Joaquin | 139 | 3.0% | 8 |

County counts show dataset representation, not earthquake risk.

## Cleanup decisions

Prioritize loading fixes, source questions, and modeling decisions. No corrections or filters are applied here.


In [22]:
sections = [
]

header_changes = "; ".join(f"`{h!r}` → `{h.strip()!r}`" for h in headers if h != h.strip()) or "No surrounding header whitespace"

issues = [
    ["Fix during loading", "Encoding and headers", f"Source uses cp1252; {header_changes}.", "Decode explicitly and normalize headers in code; preserve raw files."],
    ["Fix during typing", "Missing-value semantics", "Blanks, N/A categories, and numeric zeros have different meanings.", "Keep original tokens or missing-reason fields; convert only numeric measurement columns to nullable numeric values."],
    ["Investigate", "City labels", f"{len(summary['city_conflicts'])} facility IDs have multiple city labels (see above).", "Check against the source; record any confirmed correction in an explicit mapping. Do not overwrite all cities within a facility automatically."],
    ["Define population", "Mixed building statuses", f"{len(counts['Building Status'])} statuses; {n-len(in_service):,} records outside the exact in-service comparison subset.", "Document status/building-type inclusion rules and counts. In-service alone does not establish the final modeling population."],
    ["Define target", "Sparse Hazus labels", f"Only {present:,}/{n:,} rows have either score; {summary['coverage'][(True, True)]} have both.", "Choose a target and assess selection bias; avoid filling missing outcomes with zero or treating the two versions as interchangeable."],
    ["Confirm definitions", "Rating categories", "SPC and NPC include suffixes and extended codes.", "Preserve labels; confirm meanings before collapsing or ordinally encoding categories."],
    ["Define features", "Assessment leakage", "SPC, NPC, Hazus, and AB 1882 Notice all describe seismic assessments.", "Select predictors independently of the chosen target; review derived notices for leakage."],
    ["Protect relationships", "Facility identity and location", "Facility names and coordinates are not unique identifiers.", "Join on Perm ID and Building Nbr; consider facility-grouped evaluation when modeling."],
    ["Feature selection", "Count helper", f"Observed values: {dict(counts['Count'])}.", "If constant, exclude from predictors and retain only as a counting aid."],
]

sections.append(table(["Action", "Issue", "Evidence", "Recommended handling"], issues))



display_sections(sections)

assert SOURCE.read_bytes() == raw, "The input changed during analysis; refresh all results."


| Action | Issue | Evidence | Recommended handling |
| --- | --- | --- | --- |
| Fix during loading | Encoding and headers | Source uses cp1252; `'SPC Rating '` → `'SPC Rating'`. | Decode explicitly and normalize headers in code; preserve raw files. |
| Fix during typing | Missing-value semantics | Blanks, N/A categories, and numeric zeros have different meanings. | Keep original tokens or missing-reason fields; convert only numeric measurement columns to nullable numeric values. |
| Investigate | City labels | 2 facility IDs have multiple city labels (see above). | Check against the source; record any confirmed correction in an explicit mapping. Do not overwrite all cities within a facility automatically. |
| Define population | Mixed building statuses | 28 statuses; 1,507 records outside the exact in-service comparison subset. | Document status/building-type inclusion rules and counts. In-service alone does not establish the final modeling population. |
| Define target | Sparse Hazus labels | Only 615/4,690 rows have either score; 83 have both. | Choose a target and assess selection bias; avoid filling missing outcomes with zero or treating the two versions as interchangeable. |
| Confirm definitions | Rating categories | SPC and NPC include suffixes and extended codes. | Preserve labels; confirm meanings before collapsing or ordinally encoding categories. |
| Define features | Assessment leakage | SPC, NPC, Hazus, and AB 1882 Notice all describe seismic assessments. | Select predictors independently of the chosen target; review derived notices for leakage. |
| Protect relationships | Facility identity and location | Facility names and coordinates are not unique identifiers. | Join on Perm ID and Building Nbr; consider facility-grouped evaluation when modeling. |
| Feature selection | Count helper | Observed values: {'1': 4690}. | If constant, exclude from predictors and retain only as a counting aid. |